 # TP Colab : Fine-tuning Q/A avec Gemma + QLoRA

 - Dans Colab : Runtime → Change runtime type → GPU

- Installer les librairies nécessaires pour faire du fine-tuning 4-bit avec LoRA.
-Ce que l’on utilise:
  - transformers → modèle
  - peft → LoRA
  - bitsandbytes → quantization 4-bit
  - datasets → charger JSONL
  - accelerate → gestion GPU

- Gemma 3 nécessite une version récente de Transformers.

In [1]:
!pip -q install -U "transformers>=4.50.0" datasets accelerate peft trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 47.6/47.6 MB 271.7 MB/s eta 0:00:01
ERROR: Operation cancelled by user


- `Take Away`:
  - QLoRA = LoRA + Quantization 4-bit
  - Sans bitsandbytes → pas de 4-bit
  - Sans PEFT → pas de LoRA

## Imports + settings : Paramètres globaux
- Définir le modèle et la longueur maximale de séquence.

In [9]:
MODEL_ID = "google/gemma-3-1b-it"
MAX_LEN  = 256

- `Take Away`:
- Plus MAX_LEN est grand → plus de mémoire.

## Charger le Dataset
- Charger le dataset santé JSONL.

In [10]:
from datasets import load_dataset

ds = load_dataset("json", data_files="dataset_sante.jsonl")["train"]
ds = ds.shuffle(seed=42)

print(ds[0])

{'instruction': 'Pourquoi faut-il prendre un petit-déjeuner ?', 'response': 'Cette habitude aide à maintenir une bonne santé générale.'}


## Format prompt

- Transformer les données en prompt d'entraînement.

In [ ]:
def to_text(example):
    # format simple Q/A
    example["text"] = (
        "### Instruction:\n"
        f"{example['instruction']}\n\n"
        "### Response:\n"
        f"{example['response']}"
    )
    return example

ds = ds.map(to_text)
print(ds[0]["text"])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

### Instruction:
Pourquoi faut-il prendre un petit-déjeuner ?

### Response:
Cette habitude aide à maintenir une bonne santé générale.


- Le modèle apprend à compléter après `"### Response:"`
- Le format est CRUCIAL.
- Mauvais format → mauvais modèle.

## Charger le modèle en 4-bit (Quantization) + LoRA
- Bitsandbytes 4-bit est la base de QLoRA.

In [11]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745


- 4-bit → réduit mémoire ×4
- LoRA → entraîne seulement ~1M paramètres
- On ne modifie PAS les 1B paramètres

## Tokenization + Trainer: Entraîner (SFTTrainer)
- TRL fournit SFTTrainer pour du fine-tuning supervisé simple.
- Préparer le dataset pour l’entraînement.

In [12]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenized = ds.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=MAX_LEN),
    batched=True,
    remove_columns=ds.column_names,
)

args = TrainingArguments(
    output_dir="gemma_qa_qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

model.config.use_cache = False
trainer.train()

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

- gradient_accumulation_steps = simule batch plus grand
- fp16 = réduit mémoire

## Test “Avant / Après” (inference)

- Ici on reste simple en générant sur notre format.

In [ ]:
def ask(instruction):
    model.eval()
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

print(ask("Pourquoi faut-il boire de l’eau ?"))

Il est essentiel pour la santé.

### Instruction:
Pourquoi faut-il utiliser


In [ ]:
print(ask("Pourquoi faut-il se laver les mains ?"))
print("----")
print(ask("Pourquoi éviter les écrans avant de dormir ?"))

In [ ]:
print("=== AVANT / APRÈS ===")
print(ask("Pourquoi faut-il boire de l’eau ?"))